In [1]:
# CELL 1: CONFIG
import os, json, configparser, warnings
import numpy as np, pandas as pd, joblib
from google.colab import drive
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
warnings.filterwarnings('ignore')

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE           = "/content/drive/MyDrive/Journal_Work_2026"
MASTER_CSV_DIR = f"{BASE}/Master_Dataset_Final"
FEATURES_DIR   = f"{BASE}/Features_v2"
MODELS_DIR     = f"{BASE}/ML_Models"          # <arch>_<clf>.pkl from notebook 5
OUT_DIR        = f"{BASE}/Leakage_Audit"
LIDC_ROOT      = "/content/drive/MyDrive/Journal/DATASETS_JOURNAL/LIDC_IDRI/LIDC-IDRI"
os.makedirs(OUT_DIR, exist_ok=True)

ARCHITECTURES = ['ResNet50V2', 'DenseNet201', 'EfficientNetB3', 'EfficientNetV2L']
CLASSIFIERS   = ['SVM_Linear', 'XGBoost', 'RandomForest']
CLASS_NAMES   = ['Non-Nodule', 'Benign', 'Malignant']
N_BOOT, SEED  = 1000, 42
print("✅ Config loaded.")

Mounted at /content/drive
✅ Config loaded.


In [2]:
# CELL 2: INSTALL pylidc + compatibility fixes (same as notebook 0_LIDC-IDRI)
!pip install -q pylidc
if not hasattr(configparser, 'SafeConfigParser'):
    configparser.SafeConfigParser = configparser.ConfigParser
for _n, _t in (('int', int), ('float', float), ('bool', bool)):
    if not hasattr(np, _n):
        setattr(np, _n, _t)
with open(os.path.expanduser("~/.pylidcrc"), "w") as f:
    f.write(f"[dicom]\npath = {LIDC_ROOT}\nwarn = True\n")
import pylidc as pl
print("✅ pylidc ready.")

✅ pylidc ready.


In [3]:
# CELL 3: seriesuid -> LIDC patient_id map (metadata only, no pixel loading)
scans = pl.query(pl.Scan).all()
uid_to_patient = {s.series_instance_uid: s.patient_id for s in scans}
print(f"{len(uid_to_patient)} LIDC scans mapped.")
pd.Series(uid_to_patient, name='lidc_patient_id').rename_axis('seriesuid').to_csv(
    f"{OUT_DIR}/seriesuid_to_lidc_patient.csv")

1018 LIDC scans mapped.


In [4]:
# CELL 4: UNIFIED PATIENT ID + OVERLAP QUANTIFICATION
def load(name):
    return pd.read_csv(os.path.join(MASTER_CSV_DIR, name))

train = pd.read_csv(os.path.join(MASTER_CSV_DIR, 'train.csv'))   # originals only (pre-balancing): ids are what matter
val   = load('val.csv')
test  = load('test_corrected.csv')                                # aligned with X_test_corrected.npy

def unify(df):
    df = df.copy()
    is_luna = df['source'] == 'LUNA16'
    df['unified_id'] = df['patient_id']
    df.loc[is_luna, 'unified_id'] = df.loc[is_luna, 'patient_id'].map(uid_to_patient).fillna(df.loc[is_luna, 'patient_id'])
    df['luna_mapped'] = ~is_luna | df['patient_id'].isin(uid_to_patient)
    return df

train, val, test = unify(train), unify(val), unify(test)
print("LUNA16 ids not found in pylidc (should be 0):",
      int((~test['luna_mapped']).sum() + (~train['luna_mapped']).sum() + (~val['luna_mapped']).sum()))

trval_ids = set(train['unified_id']) | set(val['unified_id'])
test['twin_in_trainval'] = test['unified_id'].isin(trval_ids)

# Same real patient present on both sides? (unified-ID level)
for a, b, na, nb_ in [(train, test, 'train', 'test'), (val, test, 'val', 'test'), (train, val, 'train', 'val')]:
    print(f"Unified-ID overlap {na} ∩ {nb_}: {len(set(a['unified_id']) & set(b['unified_id']))} patients")

summary = (test.groupby(['source', 'label'])['twin_in_trainval']
              .agg(n='size', n_leaky='sum').assign(pct_leaky=lambda d: (100*d.n_leaky/d.n).round(1)))
print("\nTest patches whose real patient also appears in train/val:")
print(summary)
print(f"\nClean test subset: {(~test['twin_in_trainval']).sum()} / {len(test)} patches, "
      f"{test.loc[~test['twin_in_trainval'], 'unified_id'].nunique()} patients")
print("Clean class counts:", test.loc[~test['twin_in_trainval'], 'label'].value_counts().sort_index().to_dict())
summary.to_csv(f"{OUT_DIR}/overlap_summary.csv")

LUNA16 ids not found in pylidc (should be 0): 0
Unified-ID overlap train ∩ test: 80 patients
Unified-ID overlap val ∩ test: 18 patients
Unified-ID overlap train ∩ val: 80 patients

Test patches whose real patient also appears in train/val:
                 n  n_leaky  pct_leaky
source label                          
LIDC   1       168       63       37.5
       2        59       26       44.1
LNDb   0        75        0        0.0
       1        13        0        0.0
       2        96        0        0.0
LUNA16 0      2253     1349       59.9

Clean test subset: 1226 / 2664 patches, 117 patients
Clean class counts: {0: 979, 1: 118, 2: 129}


In [5]:
# CELL 5: RE-EVALUATE THE 12 TRAINED CLASSIFIERS (no retraining) + CLUSTER BOOTSTRAP
y_all   = test['label'].values
groups  = test['unified_id'].values
clean   = ~test['twin_in_trainval'].values

def macro_f1(y, p): return f1_score(y, p, average='macro', labels=[0,1,2], zero_division=0)

def cluster_bootstrap(y, pred, g, metric, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    codes, uniq = pd.factorize(g)
    idx_by = [np.where(codes == i)[0] for i in range(len(uniq))]
    vals = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(uniq), len(uniq))
        ii = np.concatenate([idx_by[p] for p in pick])
        vals.append(metric(y[ii], pred[ii]))
    return np.percentile(vals, [2.5, 97.5])

rows, preds_store = [], {}
for arch in ARCHITECTURES:
    X = np.load(f"{FEATURES_DIR}/{arch}/X_test_corrected.npy")
    y_chk = np.load(f"{FEATURES_DIR}/{arch}/y_test_corrected.npy")
    assert len(X) == len(test) and np.array_equal(y_chk, y_all), f"{arch}: test_corrected.csv not aligned with X_test_corrected"
    for clf in CLASSIFIERS:
        path = f"{MODELS_DIR}/{arch}_{clf}.pkl"
        if not os.path.exists(path):
            print("missing", path); continue
        pred = joblib.load(path).predict(X)
        preds_store[(arch, clf)] = pred
        row = {'arch': arch, 'clf': clf}
        for tag, m in (('full', np.ones(len(test), bool)), ('clean', clean)):
            acc, f1 = accuracy_score(y_all[m], pred[m]), macro_f1(y_all[m], pred[m])
            lo_a, hi_a = cluster_bootstrap(y_all[m], pred[m], groups[m], accuracy_score)
            lo_f, hi_f = cluster_bootstrap(y_all[m], pred[m], groups[m], macro_f1)
            row.update({f'acc_{tag}': round(100*acc, 2), f'acc_{tag}_CI': f"[{100*lo_a:.1f}, {100*hi_a:.1f}]",
                        f'f1_{tag}': round(100*f1, 2),  f'f1_{tag}_CI':  f"[{100*lo_f:.1f}, {100*hi_f:.1f}]"})
        rows.append(row)

res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/clean_vs_full_metrics.csv", index=False)
print(res.to_string(index=False))

           arch          clf  acc_full  acc_full_CI  f1_full   f1_full_CI  acc_clean acc_clean_CI  f1_clean  f1_clean_CI
     ResNet50V2   SVM_Linear     89.45 [86.6, 91.4]    70.29 [65.9, 74.2]      86.46 [79.4, 90.1]     72.79 [68.3, 76.3]
     ResNet50V2      XGBoost     90.69 [87.7, 92.7]    72.60 [68.2, 76.9]      87.44 [80.5, 91.2]     73.92 [68.8, 77.4]
     ResNet50V2 RandomForest     90.43 [87.4, 92.5]    71.70 [67.4, 76.1]      87.11 [79.9, 90.9]     72.99 [68.3, 76.4]
    DenseNet201   SVM_Linear     88.03 [84.6, 90.4]    67.82 [63.2, 72.2]      85.32 [78.5, 89.1]     71.71 [67.2, 75.5]
    DenseNet201      XGBoost     89.64 [85.8, 92.2]    71.42 [66.3, 76.1]      86.62 [80.2, 89.9]     74.08 [69.5, 77.7]
    DenseNet201 RandomForest     89.11 [85.0, 91.7]    69.88 [65.1, 74.2]      85.97 [78.9, 89.7]     73.00 [68.7, 76.3]
 EfficientNetB3   SVM_Linear     89.45 [86.4, 91.5]    71.36 [67.1, 76.0]      86.22 [79.7, 89.8]     73.24 [68.4, 77.3]
 EfficientNetB3      XGBoost    

In [6]:
# CELL 6: SOURCE-SHORTCUT CHECK (best model by default)
BEST = ('EfficientNetB3', 'XGBoost')
pred = preds_store[BEST]
print(f"Model: {BEST}\n")
for src in ['LUNA16', 'LIDC', 'LNDb']:
    m = (test['source'] == src).values
    print(f"{src:7s} n={m.sum():5d} acc={100*accuracy_score(y_all[m], pred[m]):.1f}%  labels present: {sorted(set(y_all[m]))}")

# Benign vs Malignant only, inside each source: if the model really separates morphology,
# accuracy should stay high WITHIN a single source (where source cannot be used as a shortcut).
print("\nBenign(1) vs Malignant(2) restricted to patches whose TRUE label is 1 or 2:")
for src in ['LIDC', 'LNDb']:
    m = ((test['source'] == src) & test['label'].isin([1, 2])).values
    cm = confusion_matrix(y_all[m], pred[m], labels=[0, 1, 2])
    print(f"\n[{src}] rows=true, cols=pred (Non-Nodule, Benign, Malignant)\n", cm)
    sub = m & np.isin(pred, [1, 2])
    if sub.sum():
        print(f"   Benign-vs-Malignant accuracy when predicted a nodule class: {100*(y_all[sub]==pred[sub]).mean():.1f}%  (n={sub.sum()})")

Model: ('EfficientNetB3', 'XGBoost')

LUNA16  n= 2253 acc=96.0%  labels present: [np.int64(0)]
LIDC    n=  227 acc=74.4%  labels present: [np.int64(1), np.int64(2)]
LNDb    n=  184 acc=52.2%  labels present: [np.int64(0), np.int64(1), np.int64(2)]

Benign(1) vs Malignant(2) restricted to patches whose TRUE label is 1 or 2:

[LIDC] rows=true, cols=pred (Non-Nodule, Benign, Malignant)
 [[  0   0   0]
 [ 35 126   7]
 [  8   8  43]]
   Benign-vs-Malignant accuracy when predicted a nodule class: 91.8%  (n=184)

[LNDb] rows=true, cols=pred (Non-Nodule, Benign, Malignant)
 [[ 0  0  0]
 [ 2  3  8]
 [20  5 71]]
   Benign-vs-Malignant accuracy when predicted a nodule class: 85.1%  (n=87)


In [7]:
pred = preds_store[('EfficientNetB3', 'XGBoost')]
m = ((test['source'] == 'LNDb') & (y_all == 0)).values
print("LNDb Non-Nodule recall:", (pred[m] == 0).mean(), "| n =", m.sum())

LNDb Non-Nodule recall: 0.29333333333333333 | n = 75
